# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Obesity Classification - Playground Series S2E2

This notebook builds a classification model to predict obesity levels using health and lifestyle features. The dataset includes demographic, behavioral, and physiological data. We'll preprocess the data, train a model, evaluate its performance, and generate predictions for submission.


## Loading the Data

We begin by importing the necessary libraries and loading the datasets provided by the competition:

- `train.csv`: Contains the features and target labels used to train the model.
- `test.csv`: Contains the features for which we need to predict the target.
- `sample_submission.csv`: Provides the required format for submitting predictions.

We also preview the shape and structure of the training data to understand its contents and verify successful loading.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load the datasets
train = pd.read_csv('/kaggle/input/playground-series-s4e2/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s4e2/test.csv')
submission = pd.read_csv('/kaggle/input/playground-series-s4e2/sample_submission.csv')

# Preview the data
print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()


## Data Preprocessing

In this step, we prepare the dataset for model training:

- **Dropped the `id` column**: This column is a unique identifier and doesn't contribute to prediction. We safely removed it only if it existed.
- **Separated features and target**: The target variable is `NObeyesdad`, representing obesity levels. All other columns are used as input features.
- **Encoded categorical variables**: Applied one-hot encoding to convert categorical features into numerical format, which is required for most machine learning models.
- **Aligned test data with training data**: Ensured that the test dataset has the same columns as the training dataset by adding any missing columns and reordering them.

This preprocessing ensures that the data is clean, consistent, and ready for model training.


In [ ]:
# Drop 'id' column only if it exists
if 'id' in train.columns:
    train.drop('id', axis=1, inplace=True)

if 'id' in test.columns:
    test.drop('id', axis=1, inplace=True)

# Define the target column
target = 'NObeyesdad'

# Separate features and target
X = train.drop(target, axis=1)
y = train[target]

# Encode categorical features
X_encoded = pd.get_dummies(X, drop_first=True)
test_encoded = pd.get_dummies(test, drop_first=True)

# Align test data columns with training data
missing_cols = set(X_encoded.columns) - set(test_encoded.columns)
for col in missing_cols:
    test_encoded[col] = 0

test_encoded = test_encoded[X_encoded.columns]




## Model Training with Hyperparameter Tuning

We train a **Random Forest Classifier** with tuned hyperparameters to improve performance:

- `n_estimators=300`: Builds more trees for better averaging and stability.
- `max_depth=20`: Limits tree depth to prevent overfitting.
- `min_samples_split=5`: Requires more samples to split a node, improving generalization.
- `min_samples_leaf=2`: Ensures each leaf has enough samples to avoid noise.

These adjustments help the model balance bias and variance, potentially improving accuracy.


In [ ]:

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# Train a Random Forest classifier with tuned hyperparameters
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
model.fit(X_train, y_train)

## Model Evaluation

We evaluated the performance of our tuned Random Forest model on the validation set:

- **Validation Accuracy**: Achieved 89.5%, showing a noticeable improvement over the baseline model.
- **Classification Report**:
  - `Obesity_Type_III` achieved perfect precision and recall.
  - `Obesity_Type_II` and `Obesity_Type_I` showed strong and consistent performance.
  - Lower-performing classes like `Overweight_Level_I` and `Normal_Weight` also improved, indicating better generalization.

These metrics confirm that hyperparameter tuning enhanced the model’s ability to distinguish between multiple obesity categories. The model is now well-prepared to make final predictions on the test set.



In [ ]:
# Make predictions on the validation set
y_pred = model.predict(X_val)

# Evaluate the model
from sklearn.metrics import accuracy_score, classification_report

print("Validation Accuracy:", accuracy_score(y_val, y_pred))
print("\nClassification Report:\n", classification_report(y_val, y_pred))


## Final Predictions & Submission

We use the trained Random Forest model to generate predictions on the test dataset.

- The predictions are inserted into the sample submission file provided in the `playground-series-s4e2` folder.
- The target column `NObeyesdad` is updated with our model's predictions.
- The final submission file `submission.csv` is saved in the correct format required by Kaggle.

This file is now ready to be uploaded for evaluation on the competition leaderboard.


In [ ]:
# Load the test dataset
test_df = pd.read_csv("/kaggle/input/playground-series-s4e2/test.csv")

# Encode categorical features
test_encoded = pd.get_dummies(test_df)

# Align test columns with training columns
test_encoded = test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

# Make predictions using the trained model
test_predictions = model.predict(test_encoded)

# Load sample submission file
submission = pd.read_csv("/kaggle/input/playground-series-s4e2/sample_submission.csv")

# Insert predictions
submission['NObeyesdad'] = test_predictions

# Save submission file
submission.to_csv("submission.csv", index=False)
